# Setup

In [1]:
import os
from pathlib import Path
import statistics

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

import catboost as cb

In [2]:
INPUT_DATA_PATH = '../data/'
OUTPUT_DATA_PATH = '../ivan-output/'
SUBMISSION_PATH = '../ivan-output/test/catboost/'
MODELS = [
    'deberta_nli',
    'deberta_nli_reverse',
    'roberta_nli',
    'roberta_nli_reverse',
    't5_nli',
    't5_nli_reverse',
    'st_roberta_nli',
    'openchat',
    ]

RANDOM_SEED = 424242

Path(SUBMISSION_PATH).mkdir(parents=True, exist_ok=True)

# Load data

In [3]:
X = {
    'val-agnostic': [],
    'val-aware': [],
    'test-agnostic': [],
    'test-aware': [],
    }
y = {}

# Load X
for model in MODELS:
    # validation agnostic
    file = OUTPUT_DATA_PATH + model + '/val/' + 'val.model-agnostic.json'
    df = pd.read_json(file, orient='records')
    df.rename(columns={"label": "label_" + model, "p(Hallucination)": "p_" + model}, inplace=True)
    X['val-agnostic'].append(df)
    print(file)
    
    # validation aware
    file = OUTPUT_DATA_PATH + model + '/val/' + 'val.model-aware.json'
    df = pd.read_json(file, orient='records')
    df.rename(columns={"label": "label_" + model, "p(Hallucination)": "p_" + model}, inplace=True)
    X['val-aware'].append(df)
    print(file)
    
    # test agnostic
    file = OUTPUT_DATA_PATH + model + '/test/' + 'test.model-agnostic.json'
    df = pd.read_json(file, orient='records')
    df.set_index('id', drop=True, inplace=True)
    df.rename(columns={"label": "label_" + model, "p(Hallucination)": "p_" + model}, inplace=True)
    X['test-agnostic'].append(df)
    print(file)
    
    # test aware
    file = OUTPUT_DATA_PATH + model + '/test/' + 'test.model-aware.json'
    df = pd.read_json(file, orient='records')
    df.set_index('id', drop=True, inplace=True)
    df.rename(columns={"label": "label_" + model, "p(Hallucination)": "p_" + model}, inplace=True)
    X['test-aware'].append(df)
    print(file)

# Load y for validation agnostic
file = INPUT_DATA_PATH + 'val/val.model-agnostic.json'
df = pd.read_json(file, orient='records')
df.rename(columns={"p(Hallucination)": "p"}, inplace=True)
y['val-agnostic'] = df[['label', 'p']]
print(file)

# Load y for validation aware
file = INPUT_DATA_PATH + 'val/val.model-aware.json'
df = pd.read_json(file, orient='records')
df.rename(columns={"p(Hallucination)": "p"}, inplace=True)
y['val-aware'] = df[['label', 'p']]
print(file)

../ivan-output/deberta_nli/val/val.model-agnostic.json
../ivan-output/deberta_nli/val/val.model-aware.json
../ivan-output/deberta_nli/test/test.model-agnostic.json
../ivan-output/deberta_nli/test/test.model-aware.json
../ivan-output/deberta_nli_reverse/val/val.model-agnostic.json
../ivan-output/deberta_nli_reverse/val/val.model-aware.json
../ivan-output/deberta_nli_reverse/test/test.model-agnostic.json
../ivan-output/deberta_nli_reverse/test/test.model-aware.json
../ivan-output/roberta_nli/val/val.model-agnostic.json
../ivan-output/roberta_nli/val/val.model-aware.json
../ivan-output/roberta_nli/test/test.model-agnostic.json
../ivan-output/roberta_nli/test/test.model-aware.json
../ivan-output/roberta_nli_reverse/val/val.model-agnostic.json
../ivan-output/roberta_nli_reverse/val/val.model-aware.json
../ivan-output/roberta_nli_reverse/test/test.model-agnostic.json
../ivan-output/roberta_nli_reverse/test/test.model-aware.json
../ivan-output/t5_nli/val/val.model-agnostic.json
../ivan-output

In [4]:
# Concatenate dataframes by columns
X_ = {}
X_['val-agnostic'] = pd.concat(X['val-agnostic'], axis=1)
X_['val-aware'] = pd.concat(X['val-aware'], axis=1)
X_['test-agnostic'] = pd.concat(X['test-agnostic'], axis=1)
X_['test-aware'] = pd.concat(X['test-aware'], axis=1)
X = X_

# For X: drop columns with 'label' feature
for df_name in X:
    for column in X[df_name].columns:
        if column.startswith('label'):
            X[df_name].drop(columns=column, inplace=True)

# Print dataframe names
print('For X data:')
for df_name in X:
    print('\t', df_name)
print()
print('For y data:')
for df_name in y:
    print('\t', df_name)

For X data:
	 val-agnostic
	 val-aware
	 test-agnostic
	 test-aware

For y data:
	 val-agnostic
	 val-aware


# CatBoost Classifier
## Set parameters

In [5]:
best_params_dict = {}
best_params_dict['val-agnostic'] = {'iterations': 53, 'learning_rate': 0.0073142127765857224, 'depth': 6, 'l2_leaf_reg': 8.226392235306717, 'border_count': 126}
best_params_dict['val-aware'] = {'iterations': 135, 'learning_rate': 0.005292073861950691, 'depth': 4, 'l2_leaf_reg': 0.17954527387864858, 'border_count': 5}
for df_name in ['val-agnostic', 'val-aware']:
    print("Best Params:", df_name.ljust(15), best_params_dict[df_name])

Best Params: val-agnostic    {'iterations': 53, 'learning_rate': 0.0073142127765857224, 'depth': 6, 'l2_leaf_reg': 8.226392235306717, 'border_count': 126}
Best Params: val-aware       {'iterations': 135, 'learning_rate': 0.005292073861950691, 'depth': 4, 'l2_leaf_reg': 0.17954527387864858, 'border_count': 5}


## Predict

In [6]:
cv = StratifiedKFold(n_splits=10)

mean_accuracy_dict = {}
dfs = {'val-agnostic': [], 'val-aware': []}
for df_name in ['val-agnostic', 'val-aware']:
    best_model = cb.CatBoostClassifier(**best_params_dict[df_name])
    accuracy = []
    for i, (train_index, test_index) in enumerate(cv.split(X[df_name], y[df_name]['label'])):
        # Train
        X_train = X[df_name].loc[train_index]
        y_train = y[df_name]['label'].loc[train_index]
        best_model.fit(X_train, y_train)

        # Predict
        X_test = X[df_name].loc[test_index]
        y_test = y[df_name]['label'].loc[test_index]
        y_pred = best_model.predict(X_test)
        
        dfs[df_name].append(pd.DataFrame(y_pred, test_index))

        # Calc metric
        accuracy.append(accuracy_score(y_test, y_pred))
        
    # Calc average metric
    mean_accuracy = statistics.fmean(accuracy)
    mean_accuracy_dict[df_name] = mean_accuracy

preds_label = {}
for df_name in ['val-agnostic', 'val-aware']:
    preds_label[df_name] = pd.concat(dfs[df_name])
    print('Accuracy:', df_name.ljust(15), mean_accuracy_dict[df_name])

0:	learn: 0.6909313	total: 48.8ms	remaining: 2.54s
1:	learn: 0.6886582	total: 50ms	remaining: 1.27s
2:	learn: 0.6863109	total: 50.7ms	remaining: 845ms
3:	learn: 0.6839280	total: 51.4ms	remaining: 630ms
4:	learn: 0.6811327	total: 51.7ms	remaining: 497ms
5:	learn: 0.6791509	total: 52.3ms	remaining: 410ms
6:	learn: 0.6767822	total: 53.1ms	remaining: 349ms
7:	learn: 0.6745781	total: 53.6ms	remaining: 302ms
8:	learn: 0.6721451	total: 54.3ms	remaining: 265ms
9:	learn: 0.6698648	total: 55ms	remaining: 236ms
10:	learn: 0.6675997	total: 55.6ms	remaining: 212ms
11:	learn: 0.6659439	total: 56.2ms	remaining: 192ms
12:	learn: 0.6637919	total: 56.8ms	remaining: 175ms
13:	learn: 0.6615743	total: 57.4ms	remaining: 160ms
14:	learn: 0.6590442	total: 57.7ms	remaining: 146ms
15:	learn: 0.6571062	total: 58.3ms	remaining: 135ms
16:	learn: 0.6547775	total: 58.8ms	remaining: 125ms
17:	learn: 0.6531704	total: 59.5ms	remaining: 116ms
18:	learn: 0.6510772	total: 60.1ms	remaining: 108ms
19:	learn: 0.6490869	total

# CatBoost Regressor
## Set parameters

In [7]:
best_params_dict = {}
best_params_dict['val-agnostic'] = {'iterations': 299, 'learning_rate': 0.012445939570737728, 'depth': 4, 'l2_leaf_reg': 0.009802660250778798, 'border_count': 8}
best_params_dict['val-aware'] = {'iterations': 216, 'learning_rate': 0.01624294306086868, 'depth': 8, 'l2_leaf_reg': 0.8617220119742635, 'border_count': 116}
for df_name in ['val-agnostic', 'val-aware']:
    print("Best Params:", df_name.ljust(15), best_params_dict[df_name])

Best Params: val-agnostic    {'iterations': 299, 'learning_rate': 0.012445939570737728, 'depth': 4, 'l2_leaf_reg': 0.009802660250778798, 'border_count': 8}
Best Params: val-aware       {'iterations': 216, 'learning_rate': 0.01624294306086868, 'depth': 8, 'l2_leaf_reg': 0.8617220119742635, 'border_count': 116}


## Predict

In [8]:
cv = StratifiedKFold(n_splits=10)

mean_corr_dict = {}
dfs = {'val-agnostic': [], 'val-aware': []}
for df_name in ['val-agnostic', 'val-aware']:
    best_model = cb.CatBoostRegressor(**best_params_dict[df_name])
    correlation = []
    for i, (train_index, test_index) in enumerate(cv.split(X[df_name], y[df_name]['label'])):
        # Train
        X_train = X[df_name].loc[train_index]
        y_train = y[df_name]['p'].loc[train_index]
        best_model.fit(X_train, y_train)

        # Predict
        X_test = X[df_name].loc[test_index]
        y_test = y[df_name]['p'].loc[test_index]
        y_pred = best_model.predict(X_test)

        # Postprocessing
        np.clip(y_pred, 0.0, 1.0, out=y_pred)

        dfs[df_name].append(pd.DataFrame(y_pred, test_index))
        
        # Calc metric
        correlation.append(spearmanr(y_pred, y_test)[0])
        
    # Calc average metric
    mean_corr = statistics.fmean(correlation)
    mean_corr_dict[df_name] = mean_corr

preds_p = {}
for df_name in ['val-agnostic', 'val-aware']:
    preds_p[df_name] = pd.concat(dfs[df_name])
    print('Correlation:', df_name.ljust(15), mean_corr_dict[df_name])

0:	learn: 0.3501016	total: 1.24ms	remaining: 369ms
1:	learn: 0.3475526	total: 1.7ms	remaining: 252ms
2:	learn: 0.3452609	total: 2.01ms	remaining: 198ms
3:	learn: 0.3428119	total: 2.32ms	remaining: 171ms
4:	learn: 0.3404937	total: 2.67ms	remaining: 157ms
5:	learn: 0.3382515	total: 3ms	remaining: 146ms
6:	learn: 0.3362002	total: 3.3ms	remaining: 138ms
7:	learn: 0.3341501	total: 3.62ms	remaining: 132ms
8:	learn: 0.3320148	total: 3.93ms	remaining: 127ms
9:	learn: 0.3300488	total: 4.24ms	remaining: 123ms
10:	learn: 0.3279519	total: 4.55ms	remaining: 119ms
11:	learn: 0.3257364	total: 4.85ms	remaining: 116ms
12:	learn: 0.3235549	total: 5.14ms	remaining: 113ms
13:	learn: 0.3215208	total: 5.45ms	remaining: 111ms
14:	learn: 0.3195095	total: 5.76ms	remaining: 109ms
15:	learn: 0.3175457	total: 6.07ms	remaining: 107ms
16:	learn: 0.3157129	total: 6.39ms	remaining: 106ms
17:	learn: 0.3139621	total: 6.68ms	remaining: 104ms
18:	learn: 0.3121981	total: 7ms	remaining: 103ms
19:	learn: 0.3104261	total: 7.

# Form dataframes

In [16]:
preds = {}
for df_name in ['val-agnostic', 'val-aware']:
    preds[df_name] = pd.DataFrame({
        'label': preds_label[df_name][0],
        'p(Hallucination)': preds_p[df_name][0],
        })

preds['val-agnostic'].head(40)

,label,p(Hallucination)
0,Not Hallucination,0.243588
1,Hallucination,0.807824
2,Hallucination,0.697204
3,Not Hallucination,0.445081
4,Not Hallucination,0.324257
5,Hallucination,0.622410
6,Not Hallucination,0.447360
7,Hallucination,0.862249
8,Hallucination,0.846152
9,Hallucination,0.632920


In [17]:
preds['val-aware'].head(40)

,label,p(Hallucination)
0,Hallucination,0.799351
1,Not Hallucination,0.353245
2,Hallucination,0.593460
3,Hallucination,0.564494
4,Hallucination,0.632992
5,Not Hallucination,0.351862
6,Hallucination,0.816110
7,Not Hallucination,0.220252
8,Not Hallucination,0.291037
9,Hallucination,0.755137


## Save to json

In [17]:
# for task in ['agnostic', 'aware']:
#     json_file_path = SUBMISSION_PATH + 'test.model-' + task + '.json'

#     # Check if file exists
#     assert not os.path.exists(json_file_path), 'File already exists.'

#     # Save to .json file
#     submission[task].to_json(
#         path_or_buf=json_file_path,
#         orient='records',
#     )

## Check file format

In [18]:
!python ../check_output.py {SUBMISSION_PATH}

all clear!
